# ⚡ Energy Demand Forecasting for Smart Grids
### Deep Neural Network (DNN) — End-to-End Pipeline

This notebook forecasts hourly electricity demand using weather and energy data from Spain.  
Models compared: **Persistence baseline**, **Linear Regression**, and a **Keras DNN**.

In [1]:
# End-to-end: data -> preprocess -> DNN (Keras) -> compare baselines -> save bundle
# Works in Colab and locally. Change file paths at top if needed.

import os, zipfile, math, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer   # 
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping


## ⚙️ Configuration — File Paths

In [2]:
# ---------------------------
# USER CHANGEABLE PARAMETERS
# ---------------------------
ENERGY_PATH = "energy_dataset.csv"
WEATHER_PATH = "weather_features.csv"

OUTPUT_DIR = "demand_forecasting_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


## 🔧 Helper Functions

In [3]:
# ---------------------------
# Helper functions
# ---------------------------
def read_csv_flex(path):
    for enc in [None, "utf-8", "utf-8-sig", "latin-1"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            continue
    return pd.read_excel(path)

def find_time_col(df):
    exact = {"timestamp","datetime","date_time","time","dt_iso","ds","date"}
    for c in df.columns:
        if c.lower() in exact:
            return c
    for c in df.columns:
        lc = c.lower()
        if "time" in lc or "date" in lc or lc in ("dt","ts"):
            return c
    return df.columns[0]

def find_target_col(df):
    prefer = ["total load actual","demand","load","consumption","electricity_demand","target","y"]
    for p in prefer:
        for c in df.columns:
            if c.lower() == p:
                return c
    parts = ["demand","load","consum","power"]
    for c in df.columns:
        lc = c.lower()
        if any(p in lc for p in parts):
            return c
    num = df.select_dtypes(include=[np.number]).columns.tolist()
    return num[-1] if num else df.columns[-1]

def rmse(y_true,y_pred): 
    return math.sqrt(mean_squared_error(y_true,y_pred))


## 📥 Load & Preprocess Data

In [4]:
# ---------------------------
# Load & preprocess
# ---------------------------
edf = read_csv_flex(ENERGY_PATH)
wdf = read_csv_flex(WEATHER_PATH)

time_col_e = find_time_col(edf)
time_col_w = find_time_col(wdf)
target_col = find_target_col(edf)

print(f"Detected: energy time={time_col_e} / weather time={time_col_w} / target={target_col}")

edf = edf.dropna(subset=[time_col_e]).copy()
wdf = wdf.dropna(subset=[time_col_w]).copy()
edf[time_col_e] = pd.to_datetime(edf[time_col_e], errors="coerce", utc=True).dt.tz_localize(None)
wdf[time_col_w] = pd.to_datetime(wdf[time_col_w], errors="coerce", utc=True).dt.tz_localize(None)
edf = edf.sort_values(time_col_e)
wdf = wdf.sort_values(time_col_w)

edf = edf.rename(columns={time_col_e: "timestamp"})
wdf = wdf.rename(columns={time_col_w: "timestamp"})
edf["timestamp"] = pd.to_datetime(edf["timestamp"]).dt.floor("H")
wdf["timestamp"] = pd.to_datetime(wdf["timestamp"]).dt.floor("H")

merged = pd.merge_asof(edf, wdf, on="timestamp", direction="nearest", tolerance=pd.Timedelta("30min"))
if merged[wdf.columns.drop("timestamp")].isna().mean().mean() > 0.5:
    merged = pd.merge(edf, wdf, on="timestamp", how="left")

numeric_cols = merged.select_dtypes(include=[np.number]).columns.tolist()
if target_col not in merged.columns:
    target_col = numeric_cols[-1]

print("Using target:", target_col)

# interpolate features only
for c in numeric_cols:
    if c != target_col:
        merged[c] = merged[c].interpolate(limit=3).fillna(method="bfill").fillna(method="ffill")

merged = merged.dropna(subset=[target_col]).reset_index(drop=True)


## 🛠️ Feature Engineering

In [5]:
# ---------------------------
# Feature engineering
# ---------------------------
df = merged.copy()
ts = pd.to_datetime(df["timestamp"])
df["hour"] = ts.dt.hour
df["dayofweek"] = ts.dt.dayofweek
df["month"] = ts.dt.month
df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)
df["season"] = df["month"].map({12:0,1:0,2:0,3:1,4:1,5:1,6:2,7:2,8:2,9:3,10:3,11:3})
df["lag1_"+target_col] = df[target_col].shift(1)
df = df.dropna(subset=["lag1_"+target_col]).reset_index(drop=True)

# split
n = len(df)
i_train = int(0.70 * n)
i_val = int(0.85 * n)
train, val, test = df.iloc[:i_train], df.iloc[i_train:i_val], df.iloc[i_val:]
print(f"Rows: total={n}, train={len(train)}, val={len(val)}, test={len(test)}")

feature_cols = [c for c in df.columns if c not in {"timestamp", target_col} and pd.api.types.is_numeric_dtype(df[c])]
print("Features used:", feature_cols)

X_train, X_val, X_test = train[feature_cols], val[feature_cols], test[feature_cols]
y_train, y_val, y_test = train[target_col], val[target_col], test[target_col]

# ⭐ FIX: impute any leftover NaNs
imputer = SimpleImputer(strategy="mean")
X_train = imputer.fit_transform(X_train)
X_val   = imputer.transform(X_val)
X_test  = imputer.transform(X_test)

scaler = StandardScaler()
X_train_s, X_val_s, X_test_s = scaler.fit_transform(X_train), scaler.transform(X_val), scaler.transform(X_test)


## 📊 Baseline Models

In [6]:
# ---------------------------
# Baselines
# ---------------------------
metrics = []

metrics.append(("Persistence","Val", mean_absolute_error(y_val,val["lag1_"+target_col]), rmse(y_val,val["lag1_"+target_col])))
metrics.append(("Persistence","Test", mean_absolute_error(y_test,test["lag1_"+target_col]), rmse(y_test,test["lag1_"+target_col])))

lin = LinearRegression().fit(X_train, y_train)
metrics.append(("LinearRegression","Val", mean_absolute_error(y_val,lin.predict(X_val)), rmse(y_val,lin.predict(X_val))))
metrics.append(("LinearRegression","Test", mean_absolute_error(y_test,lin.predict(X_test)), rmse(y_test,lin.predict(X_test))))


## 🧠 Deep Neural Network (Keras)

In [7]:
# ---------------------------
# Keras DNN
# ---------------------------
tf.random.set_seed(42)
dnn = Sequential([
    Dense(128, activation="relu", input_shape=(X_train_s.shape[1],)),
    Dropout(0.2),
    Dense(64, activation="relu"),
    Dropout(0.1),
    Dense(1)
])
dnn.compile(optimizer="adam", loss="mse", metrics=["mae"])
es = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)

history = dnn.fit(X_train_s, y_train, validation_data=(X_val_s,y_val),
                  epochs=200, batch_size=256, callbacks=[es], verbose=1)

metrics.append(("DNN_Keras","Val", mean_absolute_error(y_val,dnn.predict(X_val_s).ravel()), rmse(y_val,dnn.predict(X_val_s).ravel())))
metrics.append(("DNN_Keras","Test", mean_absolute_error(y_test,dnn.predict(X_test_s).ravel()), rmse(y_test,dnn.predict(X_test_s).ravel())))

metrics_df = pd.DataFrame(metrics, columns=["Model","Split","MAE","RMSE"])
print(metrics_df)


## 📈 Visualizations

In [8]:
# ---------------------------
# Visuals
# ---------------------------
plt.figure(figsize=(10,4))
plt.plot(df["timestamp"].iloc[-500:], df[target_col].iloc[-500:])
plt.title("Recent Demand Trend"); plt.xlabel("Time"); plt.ylabel(target_col)
plt.tight_layout(); plt.show()

plt.figure(figsize=(6,4))
plt.hist(df[target_col], bins=40)
plt.title("Target Distribution"); plt.xlabel(target_col); plt.ylabel("Count")
plt.tight_layout(); plt.show()

plt.figure(figsize=(6,4))
plt.plot(df.groupby("hour")[target_col].mean(), marker="o")
plt.title("Average Demand by Hour"); plt.xlabel("Hour"); plt.ylabel(target_col)
plt.tight_layout(); plt.show()

plt.figure(figsize=(10,4))
nplot = min(300,len(test))
plt.plot(test["timestamp"].iloc[:nplot], y_test.iloc[:nplot], label="Actual")
plt.plot(test["timestamp"].iloc[:nplot], dnn.predict(X_test_s)[:nplot], label="DNN Pred")
plt.legend(); plt.title("DNN: Actual vs Predicted (Test sample)")
plt.tight_layout(); plt.show()


## 💾 Save Outputs

In [9]:
# ---------------------------
# Save outputs
# ---------------------------
metrics_df.to_csv(os.path.join(OUTPUT_DIR,"model_metrics.csv"), index=False)
df.to_csv(os.path.join(OUTPUT_DIR,"cleaned_merged_energy_weather.csv"), index=False)

with open(os.path.join(OUTPUT_DIR,"README_demand_forecasting.txt"),"w") as f:
    f.write(f"Target: {target_col}\nFeatures: {feature_cols}\n\nMetrics:\n{metrics_df}")

with zipfile.ZipFile(os.path.join(OUTPUT_DIR,"demand_forecasting_bundle.zip"), 'w') as zf:
    for fname in ["model_metrics.csv","README_demand_forecasting.txt","cleaned_merged_energy_weather.csv"]:
        zf.write(os.path.join(OUTPUT_DIR,fname), arcname=fname)

print("\nAll artifacts saved in:", OUTPUT_DIR)
